# 24 — Part 7's stay-home counterfactual, corrected for recall of the 2020 vote

Notebook 10 applies CES 2024 stay-home rates by state and **recalled** 2020 vote (TargetSmart-validated 2024 turnout, matched respondents) to actual 2020 totals, asking whether returning every 2020 voter who stayed home in 2024 would flip a state. Here each matched respondent is given fractional membership in "reported Biden in 2020" and "reported Trump in 2020" using the CES panel's probabilities by recalled 2020 status × 2024 outcome (Harris / Trump / third party / voted, no answer / stayed home), so recalled non-voters who had in fact voted enter the rates. Everything else — sample, states, weights, the 2020 totals — is notebook 10's. Intervals: 500 bootstrap resamples of the panel — they cover uncertainty in the correction only; notebook 10's sampling interval on the cross-section still applies. Fixed before estimation.

In [1]:
import numpy as np, pandas as pd, warnings
from pathlib import Path
warnings.filterwarnings('ignore'); pd.set_option('display.width', 220)
RAW = Path('../data/raw'); TABLES = Path('../outputs/tables')
def wmean(x, w):
    m = x.notna() & w.notna(); return np.average(x[m], weights=w[m]) if m.any() else np.nan
REC = {1: 'Biden', 2: 'Trump', 3: 'other', 4: 'other', 5: 'other', 6: 'did not vote'}
def outcome(ts, vote):
    o = np.select([vote == 1, vote == 2, vote.between(3, 7)], ['Harris', 'Trump', 'third'], 'voted, no answer')
    return np.where(ts.notna() & (ts != 7), o, 'stay')
# cross-section, as notebook 10 but keeping every recalled status
raw = pd.read_csv(RAW / 'ces' / 'CCES24_Common_OUTPUT_vv_topost_final.csv', usecols=['inputstate', 'TS_g2024', 'TS_voterstatus', 'vvweight', 'vvweight_post', 'commonweight', 'presvote20post', 'CC24_410'], low_memory=False)
xs = raw[(raw.TS_voterstatus.notna() | raw.TS_g2024.notna())].copy()
xs['r'] = xs.presvote20post.map(REC); xs = xs[xs.r.notna()].copy()
xs['o'] = outcome(xs.TS_g2024, xs.CC24_410); xs['stay'] = (xs.o == 'stay').astype(float)
xs['one'] = 1.0  # raw vvweight, as notebook 10
# panel: same definitions, truth = 2020 report
p = pd.read_csv(RAW / 'ces_panel' / 'merged_recontact_2024_vv.csv', usecols=['TS_g2024', 'TS_voterstatus_24', 'presvote20post_24', 'CC24_410', 'CC20_410', 'commonpostweight_24', 'commonweight_24'], low_memory=False)
p = p[p.TS_voterstatus_24.notna() | p.TS_g2024.notna()].copy()
p['r'] = p.presvote20post_24.map(REC); p = p[p.r.notna()].copy()
p['o'] = outcome(p.TS_g2024, p.CC24_410); p['wt'] = p.commonpostweight_24.fillna(p.commonweight_24)
p['tB'] = (p.CC20_410 == 1).astype(float); p['tT'] = (p.CC20_410 == 2).astype(float)
def pis(P):
    return {k: (np.average(g.tB, weights=g.wt), np.average(g.tT, weights=g.wt), len(g)) for k, g in P.groupby(['r', 'o'])}
PI = pis(p)
print('Panel P(reported Biden), P(reported Trump) by recalled status x 2024 outcome:')
print(pd.DataFrame(PI, index=['P_Biden', 'P_Trump', 'N']).T.round(3).to_string())

Panel P(reported Biden), P(reported Trump) by recalled status x 2024 outcome:
                               P_Biden  P_Trump       N
Biden        Harris              0.982    0.002  1415.0
             Trump               0.750    0.250    25.0
             stay                0.989    0.005    95.0
             third               0.770    0.000    14.0
             voted, no answer    0.921    0.002   966.0
Trump        Harris              0.071    0.903    29.0
             Trump               0.000    0.974  1361.0
             stay                0.000    0.992    81.0
             third               0.000    1.000     8.0
             voted, no answer    0.003    0.960   465.0
did not vote Harris              0.474    0.001    54.0
             Trump               0.019    0.366    59.0
             stay                0.024    0.006    90.0
             third               0.000    0.000     3.0
             voted, no answer    0.156    0.174    58.0
other        Harris       

In [2]:
SWING_FIPS = {26: 'MICHIGAN', 37: 'NORTH CAROLINA', 4: 'ARIZONA', 13: 'GEORGIA', 42: 'PENNSYLVANIA', 55: 'WISCONSIN', 32: 'NEVADA'}
ret = pd.read_csv(RAW / 'mit_election_lab' / 'county_returns.csv')
s20 = ret[ret.year == 2020].groupby('state')[['dem_votes', 'rep_votes']].sum(); s24 = ret[ret.year == 2024].groupby('state')[['dem_votes', 'rep_votes']].sum()
def cf(xs, PI, recall=False):
    x = xs.copy()
    if recall: x['pB'] = (x.r == 'Biden').astype(float); x['pT'] = (x.r == 'Trump').astype(float)
    else:
        x['pB'] = [PI.get((r, o), (0, 0, 0))[0] for r, o in zip(x.r, x.o)]; x['pT'] = [PI.get((r, o), (0, 0, 0))[1] for r, o in zip(x.r, x.o)]
    rows = {}
    for f, st in list(SWING_FIPS.items()) + [(None, 'NATIONAL')]:
        s = x if f is None else x[x.inputstate == f]
        marg = (s24.dem_votes.sum() - s24.rep_votes.sum()) if f is None else s24.loc[st, 'dem_votes'] - s24.loc[st, 'rep_votes']
        D20 = s20.dem_votes.sum() if f is None else s20.loc[st, 'dem_votes']; R20 = s20.rep_votes.sum() if f is None else s20.loc[st, 'rep_votes']
        row = {'margin_2024': marg}
        for w in ['one', 'commonweight', 'vvweight']:
            sd = wmean(s.stay, s[w] * s.pB); sr = wmean(s.stay, s[w] * s.pT)
            row[f'net_{w}'] = D20 * sd - R20 * sr; row[f'stayB_{w}'] = sd; row[f'stayT_{w}'] = sr
        rows[st] = row
    return pd.DataFrame(rows).T
pub = cf(xs, PI, recall=True); cor = cf(xs, PI)
rng = np.random.default_rng(24); bs = [cf(xs, pis(p.sample(len(p), replace=True, random_state=rng.integers(1e9))))[['net_one', 'net_commonweight', 'net_vvweight']] for _ in range(500)]
lo = pd.concat(bs).groupby(level=0).quantile(0.025); hi = pd.concat(bs).groupby(level=0).quantile(0.975)
out = pd.DataFrame({'margin_2024': pub.margin_2024})
for w in ['one', 'commonweight', 'vvweight']:
    out[f'recall_{w}'] = pub[f'net_{w}']; out[f'corrected_{w}'] = cor[f'net_{w}']; out[f'lo_{w}'] = lo[f'net_{w}']; out[f'hi_{w}'] = hi[f'net_{w}']
    out[f'flips_{w}'] = out.margin_2024 + out[f'corrected_{w}'] > 0
print((out / 1).round(0).to_string())
print('\nStay-home rates, recall -> corrected (unweighted):')
print(pd.DataFrame({'Biden recall': pub.stayB_one, 'Biden corrected': cor.stayB_one, 'Trump recall': pub.stayT_one, 'Trump corrected': cor.stayT_one}).round(3).to_string())
out.to_csv(TABLES / 'stayhome_counterfactual_recall_corrected.csv')

                margin_2024  recall_one  corrected_one    lo_one    hi_one  flips_one  recall_commonweight  corrected_commonweight  lo_commonweight  hi_commonweight  flips_commonweight  recall_vvweight  corrected_vvweight  lo_vvweight  hi_vvweight  flips_vvweight
MICHIGAN           -80103.0     20313.0        23055.0   12900.0   36029.0        0.0              51069.0                 57352.0          45481.0          77568.0                 0.0         182835.0            181468.0     170933.0     193570.0             1.0
NORTH CAROLINA    -183048.0     58566.0        58251.0   51275.0   65720.0        0.0             113455.0                114382.0         108478.0         121383.0                 0.0         127872.0            127994.0     120046.0     135778.0             0.0
ARIZONA           -187382.0     23917.0        25281.0   21049.0   30642.0        0.0              10933.0                 11519.0           8392.0          16078.0                 0.0          26544.0       